# 03 — Pattern Generation and Feature Extraction

**Team:** StellarX  
**Phase:** 3 — Neural Network / Pattern Recognition  
**Status:** ✅ Fully executable — Phase 3 implementation complete.

> **Backend note:** PyTorch is not available on Python 3.14 (no compatible wheel as of Aug 2026).  
> Phase 3 uses a **scikit-learn RandomForest classifier** operating on pairwise-distance  
> + brightness-ratio feature vectors.  The PyTorch `StarPatternModel` stub is preserved  
> in `src/models/star_pattern_model.py` for when a compatible wheel is released.

## What this notebook covers

1. Load configuration and inspect feature parameters
2. Build the feature dataset (generate → preprocess → detect → extract_features)
3. Inspect feature vectors — distributions, class scatter, PCA
4. Sky tessellation — visualise boresight-to-label mapping
5. Train a RandomForest classifier
6. Evaluate on train / validation / test splits
7. Visualise top-k predictions on sample frames
8. Save the trained classifier checkpoint
9. Phase 4 preparation notes

In [ ]:
import warnings, sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import yaml

warnings.filterwarnings('ignore')
matplotlib.use('Agg')
%matplotlib inline

with open('config.yaml') as f:
    config = yaml.safe_load(f)

feat_cfg  = config['features']
model_cfg = config['model']
eval_cfg  = config['evaluation']

MAX_N    = feat_cfg['max_stars']
N_PAIRS  = MAX_N * (MAX_N - 1) // 2
FEAT_DIM = 2 * N_PAIRS if feat_cfg['descriptor'] == 'pairwise_distances_and_ratios' else N_PAIRS

print(f'Python       : {sys.version.split()[0]}')
print(f'Max stars    : {MAX_N}')
print(f'Descriptor   : {feat_cfg["descriptor"]}')
print(f'Feature dim  : {FEAT_DIM}  ({N_PAIRS} distances + {N_PAIRS} ratios)')
print(f'Classifier   : {model_cfg["classifier_type"]}')
print(f'Sky cells    : {model_cfg.get("n_sky_cells", 500)}')

---
## 1. Build feature dataset

Runs the full Phase 1+2 pipeline in-memory for all splits.

In [ ]:
from src.preprocessing.feature_dataset import build_feature_dataset, load_feature_dataset
from pathlib import Path

DATASET_DIR = Path('data/processed/features')

if (DATASET_DIR / 'X.npy').exists():
    print('Loading cached feature dataset…')
    X_all, y_all, meta_all = load_feature_dataset(DATASET_DIR)
else:
    print('Building feature dataset (this may take a minute)…')
    X_all, y_all, meta_all = build_feature_dataset(
        config, verbose=True, save_path=DATASET_DIR
    )

# Split by metadata
splits = np.array([m['split'] for m in meta_all])
X_train, y_train = X_all[splits=='train'], y_all[splits=='train']
X_val,   y_val   = X_all[splits=='val'],   y_all[splits=='val']
X_test,  y_test  = X_all[splits=='test'],  y_all[splits=='test']

print(f'\nDataset summary:')
print(f'  Train : {len(X_train):4d} samples  |  {len(np.unique(y_train))} unique classes')
print(f'  Val   : {len(X_val):4d} samples  |  {len(np.unique(y_val))} unique classes')
print(f'  Test  : {len(X_test):4d} samples  |  {len(np.unique(y_test))} unique classes')
print(f'  Shape : {X_train.shape}  dtype={X_train.dtype}')

n_gt_stars = np.array([m['n_stars_gt'] for m in meta_all])
n_det_stars = np.array([m['n_stars_detected'] for m in meta_all])
print(f'\n  GT stars/frame  : mean={n_gt_stars.mean():.2f}  max={n_gt_stars.max()}')
print(f'  Det stars/frame : mean={n_det_stars.mean():.2f}  max={n_det_stars.max()}')
print(f'  Zero-star frames: {(n_det_stars==0).sum()} / {len(meta_all)}')

---
## 2. Feature vector inspection

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distance feature distribution (first N_PAIRS elements)
ax = axes[0]
nonzero_dist = X_train[:, :N_PAIRS]
nonzero_dist = nonzero_dist[nonzero_dist > 0].flatten()
ax.hist(nonzero_dist, bins=40, color='steelblue', edgecolor='white')
ax.set_xlabel('Normalised pairwise distance')
ax.set_ylabel('Count')
ax.set_title('Distance feature distribution\n(non-zero entries, train set)')
ax.grid(True, alpha=0.3)

# Brightness ratio distribution (last N_PAIRS elements)
ax = axes[1]
nonzero_ratio = X_train[:, N_PAIRS:]
nonzero_ratio = nonzero_ratio[nonzero_ratio > 0].flatten()
ax.hist(nonzero_ratio, bins=40, color='mediumseagreen', edgecolor='white')
ax.set_xlabel('Brightness ratio')
ax.set_ylabel('Count')
ax.set_title('Brightness ratio distribution\n(non-zero entries, train set)')
ax.grid(True, alpha=0.3)

# Stars per frame
ax = axes[2]
ax.hist(n_gt_stars, bins=range(0, n_gt_stars.max()+2), color='salmon',
        edgecolor='white', label='GT')
ax.hist(n_det_stars, bins=range(0, n_det_stars.max()+2), color='steelblue',
        edgecolor='white', alpha=0.6, label='Detected')
ax.set_xlabel('Stars per frame')
ax.set_ylabel('Frames')
ax.set_title('GT vs Detected stars per frame')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('notebooks/fig_09_feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved → notebooks/fig_09_feature_distributions.png')

---
## 3. PCA visualisation of feature space

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Use only frames that have at least 1 detected star (non-zero features)
has_stars = (n_det_stars[splits=='train'] > 0)
X_pca_in = X_train[has_stars]
y_pca_in = y_train[has_stars]

if len(X_pca_in) >= 10:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_pca_in)
    pca = PCA(n_components=2, random_state=42)
    X_2d = pca.fit_transform(X_scaled)

    fig, ax = plt.subplots(figsize=(8, 6))
    scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1],
                         c=y_pca_in, cmap='tab20', s=20, alpha=0.7)
    plt.colorbar(scatter, ax=ax, label='Sky-cell label')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title('PCA of feature vectors (train, frames with ≥1 detected star)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('notebooks/fig_10_pca_features.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Total explained variance: {sum(pca.explained_variance_ratio_)*100:.1f}%')
    print('Figure saved → notebooks/fig_10_pca_features.png')
else:
    print('Not enough non-zero-feature samples for PCA — skipping.')

---
## 4. Sky tessellation visualisation

In [ ]:
from src.models.sklearn_classifier import boresight_to_label

n_sky = config['model'].get('n_sky_cells', 500)
ra_grid  = np.linspace(0, 360, 180)
dec_grid = np.linspace(-90, 90, 90)
RA, DEC  = np.meshgrid(ra_grid, dec_grid)
labels_grid = np.vectorize(lambda r, d: boresight_to_label(r, d, n_sky))(RA, DEC)

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(labels_grid, extent=[0, 360, -90, 90], origin='lower',
               aspect='auto', cmap='tab20b', interpolation='nearest')
plt.colorbar(im, ax=ax, label='Sky-cell label')

# Overlay training boresights
train_ra  = np.array([m['boresight_ra_deg']  for m in meta_all if m['split']=='train'])
train_dec = np.array([m['boresight_dec_deg'] for m in meta_all if m['split']=='train'])
ax.scatter(train_ra, train_dec, s=8, color='white', alpha=0.6, label='Train boresights')

ax.set_xlabel('RA (deg)')
ax.set_ylabel('Dec (deg)')
ax.set_title(f'Sky tessellation — {n_sky} target cells\n'
             f'White dots = training boresights ({len(train_ra)} samples)')
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.savefig('notebooks/fig_11_sky_tessellation.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Unique labels in train set: {len(np.unique(y_train))}')
print('Figure saved → notebooks/fig_11_sky_tessellation.png')

---
## 5. Train classifier

In [ ]:
from src.models.sklearn_classifier import train_classifier, evaluate_classifier

print(f'Training {model_cfg["classifier_type"]} on {len(X_train)} samples…')
clf, train_result = train_classifier(X_train, y_train, config, seed=42)

print(f'\nTraining result:')
print(f'  Classifier    : {train_result.classifier_type}')
print(f'  Samples       : {train_result.n_train}')
print(f'  Classes       : {train_result.n_classes}')
print(f'  Feature dim   : {train_result.feature_dim}')
print(f'  Train accuracy: {train_result.train_accuracy:.4f}')
print(f'  Training time : {train_result.elapsed_sec:.2f}s')

---
## 6. Evaluate on all splits

In [ ]:
rows = []
for split_name, Xs, ys in [('train', X_train, y_train),
                             ('val',   X_val,   y_val),
                             ('test',  X_test,  y_test)]:
    if len(Xs) == 0:
        continue
    metrics = evaluate_classifier(clf, Xs, ys, config)
    rows.append({
        'split':             split_name,
        'n_samples':         metrics['n_test'],
        'top1_accuracy':     round(metrics['top1_accuracy'], 4),
        f'top{metrics["top_k"]}_accuracy': round(metrics['topk_accuracy'], 4),
        'n_above_threshold': metrics['n_above_threshold'],
        'acc_above_thresh':  round(metrics['accuracy_above_threshold']
                                   if not isinstance(metrics['accuracy_above_threshold'], float)
                                   or not __import__('math').isnan(metrics['accuracy_above_threshold'])
                                   else float('nan'), 4),
    })

df_eval = pd.DataFrame(rows)
print(df_eval.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Accuracy bar chart
ax = axes[0]
x = np.arange(len(rows))
labels_bar = [r['split'] for r in rows]
top1  = [r['top1_accuracy'] for r in rows]
topk  = [r.get(f'top{eval_cfg["top_k"]}_accuracy', r['top1_accuracy']) for r in rows]
w = 0.35
ax.bar(x - w/2, top1, w, label='Top-1', color='steelblue')
ax.bar(x + w/2, topk, w, label=f'Top-{eval_cfg["top_k"]}', color='mediumseagreen')
ax.set_xticks(x); ax.set_xticklabels(labels_bar)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy')
ax.set_title('Top-1 and Top-k Accuracy per Split')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

# Confidence distribution
ax = axes[1]
_, confs_test = clf.predict_batch(X_test) if len(X_test) else (None, np.array([]))
if len(confs_test) > 0:
    ax.hist(confs_test, bins=30, color='cornflowerblue', edgecolor='white')
    ax.axvline(eval_cfg['confidence_threshold'], color='tomato', linestyle='--',
               label=f'Threshold={eval_cfg["confidence_threshold"]}')
    ax.set_xlabel('Confidence (predicted class probability)')
    ax.set_ylabel('Frames')
    ax.set_title('Confidence Distribution (test set)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No test samples', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig('notebooks/fig_12_evaluation_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved → notebooks/fig_12_evaluation_results.png')

---
## 7. Sample frame predictions

In [ ]:
from src.models.inference import run_inference

# Show predictions for 6 test frames
n_show = min(6, len(X_test))
print(f'Sample predictions on {n_show} test frames:')
print(f'{"Frame":<8} {"GT label":<12} {"Pred label":<12} {"Conf":>8}  {"Correct":>8}  Top-3')
print('-' * 75)
for i in range(n_show):
    result = run_inference(clf, X_test[i], config)
    gt = y_test[i]
    pred_lbl = result.pattern_id or 'NONE'
    correct = '✓' if result.pattern_id == f'cell_{gt}' else '✗'
    topk_str = '  '.join([f'{p}({c:.2f})' for p, c in result.top_k_predictions[:3]])
    print(f'{i:<8} {f"cell_{gt}":<12} {pred_lbl:<12} {result.confidence:>8.4f}  {correct:>8}  {topk_str}')

---
## 8. Save classifier checkpoint

In [ ]:
import os
checkpoint_path = os.path.join(
    config['model']['checkpoint_dir'],
    config['model']['checkpoint_name']
)
clf.save(checkpoint_path)
print(f'Classifier saved → {checkpoint_path}')

# Verify round-trip
from src.models.inference import load_model
clf_loaded = load_model(checkpoint_path, config)
labels_orig,  _ = clf.predict_batch(X_test[:5] if len(X_test) >= 5 else X_train[:5])
labels_loaded, _ = clf_loaded.predict_batch(X_test[:5] if len(X_test) >= 5 else X_train[:5])
assert (labels_orig == labels_loaded).all(), 'Round-trip predictions differ!'
print('Round-trip load/predict verification: ✓ PASSED')

---
## 9. Phase 4 preparation notes

**Phase 3 summary:**

| Item | Status |
|---|---|
| `extract_features()` — pairwise distances + brightness ratios | ✅ Implemented |
| Fixed-length feature vector (90 dim for N=10 stars) | ✅ |
| Sky tessellation (`boresight_to_label`) | ✅ |
| `build_feature_dataset()` — full pipeline integration | ✅ |
| `StarPatternClassifier` (RF / KNN / MLP) — train/predict/save/load | ✅ |
| `run_inference()` — confidence-gated, top-k predictions | ✅ |
| Unit tests (90 assertions, 10 test classes) | ✅ Passing |

**Key observations and Phase 4 recommendations:**

1. **Sparse catalog**: Frames with 0 detected stars yield zero feature vectors.  
   These provide no useful discrimination signal.  
   → **First action for Phase 4:** extend the catalog to the full Hipparcos dataset.

2. **Classification accuracy**: With a sparse catalog, many training frames map to  
   the same sky cell (only ~5–15 cells populated).  Once the catalog is extended,  
   more cells will be populated and accuracy will be measurable more meaningfully.

3. **Feature representation**: Pairwise distances work well for bright, widely-  
   separated stars.  For Phase 4, consider adding triangle-based geometric  
   descriptors which are more robust to partial occlusion.

4. **Catalog matching**: Phase 4 should implement `pattern_matcher.py` — matching  
   the recognised sky-cell ID against the catalog to recover the actual star IDs  
   and their celestial coordinates.

**Next notebook:** `04_model_training.ipynb` — detailed training diagnostics.